# Problem Solving Through Search — Performance Evaluation

**Name : Dhruv Bompilwar**  
**PRN : 202401040200**  
**Roll no : 49**  
**Division : A**  
**Batch : A2**

## Objective

Evaluate different problem-solving search algorithms using **the same problem situation throughout the notebook**.

### Common Problem: 8-Queens

The goal is to place 8 queens on an 8×8 chessboard so that no two queens attack each other.

The same **8-Queens problem** is solved using:

1. Uninformed Search — BFS, DFS, Uniform Cost Search
2. Informed Search — Greedy Best-First Search, A*
3. Local Search — Hill Climbing, Simulated Annealing
4. Constraint Satisfaction — Backtracking

Using one common problem makes the performance comparison more consistent.

## Common State Representation

A state is represented as a list of 8 integers.

- Index = row number
- Value = column containing the queen
- Example: `[0, 4, 7, 5, 2, 6, 1, 3]`

The search starts with an empty board and adds one queen at a time. A state at depth `k` represents the positions of the first `k` queens.

The cost of a state is its number of attacking queen pairs. A solution has **0 conflicts**.

In [ ]:
import time
import heapq
import random
import math
import pandas as pd

N = 8

def conflicts(state):
    """Return the number of pairs of queens attacking each other."""
    count = 0

    for i in range(len(state)):
        for j in range(i + 1, len(state)):
            # Same column
            if state[i] == state[j]:
                count += 1

            # Same diagonal
            if abs(state[i] - state[j]) == abs(i - j):
                count += 1

    return count


def is_valid_partial(state):
    """Check whether a partial queen placement violates constraints."""
    for i in range(len(state)):
        for j in range(i + 1, len(state)):
            if state[i] == state[j]:
                return False

            if abs(state[i] - state[j]) == abs(i - j):
                return False

    return True


def successors(state):
    """Generate states by placing a queen in the next row."""
    if len(state) == N:
        return []

    row = len(state)
    children = []

    for col in range(N):
        new_state = state + [col]

        if is_valid_partial(new_state):
            children.append(new_state)

    return children


def display_board(state):
    """Print a chessboard representation of a solution."""
    if state is None:
        print("No solution")
        return

    for row in state:
        print(" ".join("Q" if col == row else "." for col in range(N)))


def measure(function, *args):
    start = time.perf_counter()
    result = function(*args)
    elapsed = time.perf_counter() - start
    return result, elapsed


## 1. Uninformed Search

### Breadth First Search (BFS)
BFS explores the state space level by level. Here, each level represents placing one additional queen. It does not use information about how promising a state is.

### Depth First Search (DFS)
DFS follows one branch deeply before trying another branch. It generally requires less memory than BFS, but its search order may not be efficient.

### Uniform Cost Search (UCS)
UCS expands the state with the lowest accumulated cost. Here, the cost is based on the number of conflicts in the partial state. It does not use a heuristic estimate of the remaining problem.

In [ ]:
# ------------------------------------------------------------
# UNINFORMED SEARCH ON THE 8-QUEENS PROBLEM
# ------------------------------------------------------------

def bfs_queens():
    queue = [([], 0)]
    nodes_explored = 0

    while queue:
        state, cost = queue.pop(0)
        nodes_explored += 1

        if len(state) == N and conflicts(state) == 0:
            return state, nodes_explored

        for child in successors(state):
            queue.append((child, conflicts(child)))

    return None, nodes_explored


def dfs_queens():
    stack = [([], 0)]
    nodes_explored = 0

    while stack:
        state, cost = stack.pop()
        nodes_explored += 1

        if len(state) == N and conflicts(state) == 0:
            return state, nodes_explored

        # Reverse only controls the deterministic order of exploration.
        children = successors(state)

        for child in reversed(children):
            stack.append((child, conflicts(child)))

    return None, nodes_explored


def ucs_queens():
    counter = 0
    priority_queue = [(0, counter, [])]
    visited = set()
    nodes_explored = 0

    while priority_queue:
        cost, _, state = heapq.heappop(priority_queue)
        nodes_explored += 1

        state_tuple = tuple(state)

        if state_tuple in visited:
            continue

        visited.add(state_tuple)

        if len(state) == N and conflicts(state) == 0:
            return state, nodes_explored

        for child in successors(state):
            counter += 1
            child_cost = conflicts(child)
            heapq.heappush(
                priority_queue,
                (child_cost, counter, child)
            )

    return None, nodes_explored


for name, function in [
    ("BFS", bfs_queens),
    ("DFS", dfs_queens),
    ("UCS", ucs_queens)
]:
    result, elapsed = measure(function)
    solution, nodes = result

    print(f"\n{name}")
    print("Solution:", solution)
    print("Conflicts:", conflicts(solution) if solution else "N/A")
    print("Nodes explored:", nodes)
    print(f"Time: {elapsed:.8f} seconds")


## 2. Informed Search

### Greedy Best-First Search
Greedy Best-First Search selects the state with the best heuristic value. The heuristic estimates how promising a partial queen placement is by considering its current conflicts and the remaining rows.

### A* Search
A* combines the cost already incurred `g(n)` with the heuristic estimate `h(n)`:

`f(n) = g(n) + h(n)`

It therefore considers both the current state and an estimate of future difficulty.

In [ ]:
# ------------------------------------------------------------
# INFORMED SEARCH ON THE SAME 8-QUEENS PROBLEM
# ------------------------------------------------------------

def heuristic_queens(state):
    """Lower heuristic value means a more promising state.

    It combines current conflicts with the number of rows remaining.
    """
    return conflicts(state) + (N - len(state))


def greedy_queens():
    counter = 0
    priority_queue = [(heuristic_queens([]), counter, [])]
    visited = set()
    nodes_explored = 0

    while priority_queue:
        h, _, state = heapq.heappop(priority_queue)
        nodes_explored += 1

        state_tuple = tuple(state)

        if state_tuple in visited:
            continue

        visited.add(state_tuple)

        if len(state) == N and conflicts(state) == 0:
            return state, nodes_explored

        for child in successors(state):
            counter += 1
            heapq.heappush(
                priority_queue,
                (heuristic_queens(child), counter, child)
            )

    return None, nodes_explored


def a_star_queens():
    counter = 0
    start = []
    start_g = 0

    priority_queue = [
        (start_g + heuristic_queens(start), start_g, counter, start)
    ]

    best_g = {tuple(start): 0}
    nodes_explored = 0

    while priority_queue:
        f, g, _, state = heapq.heappop(priority_queue)
        nodes_explored += 1

        state_tuple = tuple(state)

        if g > best_g.get(state_tuple, float("inf")):
            continue

        if len(state) == N and conflicts(state) == 0:
            return state, nodes_explored

        for child in successors(state):
            # Each queen placement represents one step.
            new_g = g + 1
            child_tuple = tuple(child)

            if new_g < best_g.get(child_tuple, float("inf")):
                best_g[child_tuple] = new_g
                counter += 1
                f = new_g + heuristic_queens(child)

                heapq.heappush(
                    priority_queue,
                    (f, new_g, counter, child)
                )

    return None, nodes_explored


for name, function in [
    ("Greedy Best-First", greedy_queens),
    ("A*", a_star_queens)
]:
    result, elapsed = measure(function)
    solution, nodes = result

    print(f"\n{name}")
    print("Solution:", solution)
    print("Conflicts:", conflicts(solution) if solution else "N/A")
    print("Nodes explored:", nodes)
    print(f"Time: {elapsed:.8f} seconds")


## 3. Local Search

Both algorithms solve the **same 8-Queens problem** but work with complete board configurations.

### Hill Climbing
Starts from a random board and repeatedly moves to a neighbouring board with fewer conflicts. It is fast and memory-efficient, but can become trapped in a local minimum.

### Simulated Annealing
Similar to Hill Climbing, but occasionally accepts a worse state. The probability of accepting a worse move decreases as the temperature decreases, helping the algorithm escape local minima.

In [ ]:
# ------------------------------------------------------------
# LOCAL SEARCH ON THE SAME 8-QUEENS PROBLEM
# ------------------------------------------------------------

def random_neighbour(state):
    neighbour = state.copy()
    col = random.randrange(N)
    new_row = random.randrange(N)
    neighbour[col] = new_row
    return neighbour


def hill_climbing(max_restarts=100):
    states_checked = 0

    for _ in range(max_restarts):
        state = [random.randrange(N) for _ in range(N)]

        while True:
            current = conflicts(state)

            if current == 0:
                return state, states_checked

            neighbours = []

            for col in range(N):
                for row in range(N):
                    if row != state[col]:
                        new_state = state.copy()
                        new_state[col] = row
                        neighbours.append(new_state)

            states_checked += len(neighbours)

            best = min(neighbours, key=conflicts)

            if conflicts(best) >= current:
                break

            state = best

    return state, states_checked


def simulated_annealing(max_iterations=10000):
    state = [random.randrange(N) for _ in range(N)]
    states_checked = 0

    for t in range(1, max_iterations + 1):
        current = conflicts(state)

        if current == 0:
            return state, states_checked

        temperature = max(0.01, 10 * (0.995 ** t))

        neighbour = random_neighbour(state)
        new_cost = conflicts(neighbour)

        states_checked += 1

        delta = new_cost - current

        if delta < 0:
            state = neighbour
        else:
            probability = math.exp(-delta / temperature)

            if random.random() < probability:
                state = neighbour

    return state, states_checked


random.seed(42)

result, elapsed = measure(hill_climbing)
solution, states = result

print("Hill Climbing")
print("Solution:", solution)
print("Conflicts:", conflicts(solution))
print("States checked:", states)
print(f"Time: {elapsed:.8f} seconds")

random.seed(42)

result, elapsed = measure(simulated_annealing)
solution, states = result

print("\nSimulated Annealing")
print("Solution:", solution)
print("Conflicts:", conflicts(solution))
print("States checked:", states)
print(f"Time: {elapsed:.8f} seconds")


## 4. Constraint Satisfaction Problem (CSP)

### Backtracking Search

The same **8-Queens problem** is treated as a CSP.

- **Variables:** Rows 1–8
- **Domain:** Columns 1–8
- **Constraints:** No two queens may share the same column or diagonal.

Backtracking assigns one queen at a time and immediately rejects an assignment when it violates a constraint. This prunes large parts of the search space.

In [ ]:
# ------------------------------------------------------------
# CONSTRAINT SATISFACTION: BACKTRACKING
# ON THE SAME 8-QUEENS PROBLEM
# ------------------------------------------------------------

def backtracking_queens():
    board = [-1] * N
    states_checked = 0

    def is_safe(row, col):
        for previous_row in range(row):
            previous_col = board[previous_row]

            if previous_col == col:
                return False

            if abs(previous_col - col) == abs(previous_row - row):
                return False

        return True

    def backtrack(row):
        nonlocal states_checked

        if row == N:
            return True

        for col in range(N):
            states_checked += 1

            if is_safe(row, col):
                board[row] = col

                if backtrack(row + 1):
                    return True

                board[row] = -1

        return False

    success = backtrack(0)

    return (board if success else None), states_checked


result, elapsed = measure(backtracking_queens)
solution, states = result

print("Backtracking CSP")
print("Solution:", solution)
print("Conflicts:", conflicts(solution) if solution else "N/A")
print("States checked:", states)
print(f"Time: {elapsed:.8f} seconds")


## 5. Final Performance Comparison

The next cell executes all algorithms on the **same 8-Queens problem** and creates a common comparison table.

For BFS, DFS, UCS, Greedy Best-First, and A*, the number of **search states expanded** is measured.

For Hill Climbing and Simulated Annealing, the number of **candidate states checked** is measured.

For Backtracking, the number of **variable assignments tested** is measured.

In [ ]:
# Run all algorithms on the same 8-Queens problem

algorithms = [
    ("BFS", "Uninformed", bfs_queens),
    ("DFS", "Uninformed", dfs_queens),
    ("UCS", "Uninformed", ucs_queens),
    ("Greedy Best-First", "Informed", greedy_queens),
    ("A*", "Informed", a_star_queens),
    ("Hill Climbing", "Local Search", hill_climbing),
    ("Simulated Annealing", "Local Search", simulated_annealing),
    ("Backtracking", "CSP", backtracking_queens)
]

comparison = []

for name, category, function in algorithms:
    random.seed(42)

    result, elapsed = measure(function)
    solution, states = result

    comparison.append([
        name,
        category,
        solution,
        conflicts(solution) if solution else None,
        states,
        elapsed
    ])

df = pd.DataFrame(
    comparison,
    columns=[
        "Algorithm",
        "Category",
        "Solution",
        "Conflicts",
        "States Checked / Explored",
        "Execution Time (seconds)"
    ]
)

display(df)


## 6. Example Solution Board

All algorithms are attempting to solve the same problem. The following code displays one valid 8-Queens solution obtained from the search.

In [ ]:
solution, _ = backtracking_queens()

print("8-Queens Solution:")
display_board(solution)
print("\nNumber of conflicts:", conflicts(solution))


## 7. Conclusion

All algorithms in this notebook solve the **same 8-Queens problem**, making the comparison more meaningful.

- **BFS** systematically explores the problem level by level but can require many states and substantial memory.
- **DFS** generally uses less memory and can find a solution quickly depending on the search order, but it does not guarantee an efficient search.
- **UCS** prioritizes the lowest accumulated cost and is useful when path costs differ.
- **Greedy Best-First Search** uses heuristic information to guide the search and can reduce exploration, but it does not guarantee optimality.
- **A\*** combines actual cost and heuristic information and is generally more informed than uninformed methods.
- **Hill Climbing** uses very little memory and can be fast, but it may become trapped in local minima.
- **Simulated Annealing** can escape local minima by accepting some worse moves during the early stages of the search.
- **Backtracking** is particularly suitable for the 8-Queens CSP because constraint violations can be detected immediately, allowing large portions of the search space to be pruned.

### Overall Observation

The best algorithm depends on the structure of the problem. Uninformed methods do not require domain knowledge, informed methods benefit from heuristics, local-search methods are useful for optimization with limited memory, and CSP methods exploit explicit constraints to reduce the search space.